## Pipeline Element Influence
Analyse der Pipeline-Komponenten fuer Geschlechtsmodelle.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score
from joblib import dump

from FIT_python.data_split_and_summary.datasplit_and_summary_wraper import SplitWrapper
from FIT_python.data_split_and_summary.summary_data_wrapper import SummaryWrapper
from FIT_python.general_pipeline_steps.feature_selection_wrapper import FeatureSelectionTransformer
from FIT_python.general_pipeline_steps.outlier_wrapper import OutlierCleanerTransformer
from FIT_python.general_pipeline_steps.feature_scaler_wrapper import FeatureScalerTransformer
from FIT_python.general_pipeline_steps.dimensionality_reduction_wrapper import DimensionalityReducerTransformer
from FIT_python.general_pipeline_steps.models import MODELS
from FIT_python.data_split_and_summary.transform_wrapper import NumericTransformer
from FIT_python.plot_style import apply_style
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer

apply_style()

EXP_DIR = Path("experiments/experiments/pipeline_element_influence")
EXP_DIR.mkdir(parents=True, exist_ok=True)


## Create Splits
Alle Rohdaten werden geladen und in Trainings-, Test- und Inferenz-Splits unter EXP_DIR/splits gespeichert.

In [ ]:
split_dir = EXP_DIR / "splits"
SplitWrapper(output_dir=split_dir).split_all(as_csv=True)
SummaryWrapper().summarize_all(splits_dir=split_dir, output_table=EXP_DIR/"split_summary.csv", fig_dir=EXP_DIR/"split_figs")


## Bayesian Hyperparameter Search
BayesSearchCV untersucht verschiedene Pipelineelemente.

In [ ]:
results = []
search_records = []
pipe_template = Pipeline([
    ("numeric", NumericTransformer()),
    ("outlier", "passthrough"),
    ("scale", "passthrough"),
    ("select", FeatureSelectionTransformer()),
    ("reduce_pre", "passthrough"),
    ("clf", MODELS["logreg_l2"]),
])
search_space = {
    "outlier": Categorical([
        "passthrough",
        OutlierCleanerTransformer(method="clip"),
        OutlierCleanerTransformer(method="zscore"),
    ], transform="identity"),
    "scale": Categorical([
        "passthrough",
        FeatureScalerTransformer(method="standard"),
        FeatureScalerTransformer(method="robust"),
    ], transform="identity"),
    "select": Categorical([
        "passthrough",
        FeatureSelectionTransformer(method="forward", k=5),
        FeatureSelectionTransformer(method="forward", k=10),
        FeatureSelectionTransformer(method="lasso", k=10),
    ], transform="identity"),
    "reduce_pre": Categorical([
        "passthrough",
        DimensionalityReducerTransformer(method="pca", n_components=5),
        DimensionalityReducerTransformer(method="umap", n_components=5),
    ], transform="identity"),
    "clf": Categorical([
        MODELS["logreg_l2"],
        MODELS["rf_small"],
        MODELS["xgb_hist"],
    ], transform="identity"),
}

split_dir = EXP_DIR / "splits"
for sp in sorted(split_dir.iterdir()):
    if not sp.is_dir():
        continue
    train_fp = sp / "train.parquet"
    test_fp = sp / "test.parquet"
    if not train_fp.exists() or not test_fp.exists():
        continue
    train_df = pd.read_parquet(train_fp)
    test_df = pd.read_parquet(test_fp)
    num_cols = train_df.select_dtypes(include=np.number).columns
    feature_cols = [c for c in num_cols if c != "Fold"]
    X_tr = train_df[feature_cols]
    y_tr = train_df["sex"].map({"f":0,"m":1})
    X_te = test_df[feature_cols]
    y_te = test_df["sex"].map({"f":0,"m":1})
    cv = PredefinedSplit(train_df["Fold"]) if "Fold" in train_df.columns else 3
    bayes = BayesSearchCV(
        estimator=pipe_template,
        search_spaces=search_space,
        n_iter=20,
        scoring="balanced_accuracy",
        cv=cv,
        n_jobs=-1,
        random_state=42,
        refit=True,
    )
    bayes.fit(X_tr, y_tr)
    best_model = bayes.best_estimator_
    preds = best_model.predict(X_te)
    test_acc = balanced_accuracy_score(y_te, preds)
    dump(best_model, EXP_DIR / f"{sp.name}_best_model.joblib")
    pd.DataFrame(bayes.cv_results_).to_csv(EXP_DIR / f"{sp.name}_search.csv", index=False)
    results.append({
        "species": sp.name,
        "cv_best_balanced_accuracy": bayes.best_score_,
        "test_balanced_accuracy": test_acc,
    })
    search_records.append(pd.DataFrame(bayes.cv_results_).assign(species=sp.name))

results_df = pd.DataFrame(results)
results_df.to_csv(EXP_DIR / "results_summary.csv", index=False)
search_df = pd.concat(search_records, ignore_index=True)
search_df.to_csv(EXP_DIR / "search_details.csv", index=False)


## Visualisation
Nutze Hilfsfunktionen fuer einheitliche Darstellungen.

In [ ]:
from FIT_python.pipeline_sex.sex_predict_and_visualisation import plot_hyperparam_heatmap
plot_hyperparam_heatmap(search_df.rename(columns={
    "select__method": "fs_method",
    "reduce_pre__method": "reduce_pre_method",
    "mean_test_score": "cv_balanced_accuracy"}), EXP_DIR / "figs")


## LLM ready results
Die wichtigsten Kennzahlen werden in einer kompakten CSV gespeichert.

In [ ]:
llm_file = EXP_DIR / "llm_results.txt"
with llm_file.open("w", encoding="utf-8") as f:
    for row in results:
        f.write(f"{row['species']}: CV={row['cv_best_balanced_accuracy']:.3f}, Test={row['test_balanced_accuracy']:.3f}
")
llm_file
